In [1]:
import pandas as pd

In [2]:
caminho = '../data/processed/'

fato = pd.read_csv(caminho + 'fato_internacoes.csv')
dim_paciente = pd.read_csv(caminho + 'dim_paciente.csv')
dim_procedimento = pd.read_csv(caminho + 'dim_procedimento.csv')
dim_setor = pd.read_csv(caminho + 'dim_setor.csv')
dim_tempo = pd.read_csv(caminho + 'dim_tempo.csv')

In [5]:
df_unificado = fato \
    .merge(dim_paciente, on='id_paciente', how='left') \
    .merge(dim_procedimento, on='id_procedimento', how='left') \
    .merge(dim_setor, on='id_setor', how='left') \
    .merge(dim_tempo, left_on='data_entrada', right_on='data', how='left')

In [4]:
dim_tempo.columns

Index(['data', 'ano', 'mes', 'dia'], dtype='object')

In [6]:
print(df_unificado.shape)
print(df_unificado.isnull().sum())

(592, 17)
id_internacao             0
id_paciente               0
id_procedimento           0
id_setor                  0
data_entrada              0
custo_total               0
quantidade                0
tempo_internacao          0
idade                     0
sexo                      0
faixa_etaria              0
descricao_procedimento    0
setor                     0
data                      0
ano                       0
mes                       0
dia                       0
dtype: int64


In [7]:
tabela_resumo = df_unificado.groupby('setor').agg({
    'custo_total': 'sum',
    'quantidade': 'sum',
    'tempo_internacao': 'mean',
    'id_paciente': 'nunique'
}).reset_index()

tabela_resumo.columns = [
    'Setor',
    'Custo Total',
    'Qtd Procedimentos',
    'Tempo Médio Internação',
    'Qtd Pacientes'
]

tabela_resumo

,Setor,Custo Total,Qtd Procedimentos,Tempo Médio Internação,Qtd Pacientes
0,Centro Cirúrgico,332360,211,7.964286,35
1,Emergência,507690,382,8.507853,46
2,Enfermaria,308440,281,7.166667,32
3,UTI,338580,306,7.000000,39


In [8]:
tabela_pivot = pd.pivot_table(
    df_unificado,
    values='custo_total',
    index='setor',
    columns='mes',
    aggfunc='sum',
    fill_value=0
)

tabela_pivot

mes,1,2,3,4,5,6,7,8,9,10,11,12
setor,,,,,,,,,,,,
Centro Cirúrgico,0,11970,46730,830,80370,11060,43610,800,41940,27010,46670,21370
Emergência,97550,20130,51990,30030,35860,28770,29370,30010,80120,3080,40080,60700
Enfermaria,35790,37350,29450,7820,18250,48780,33850,4850,6890,300,48100,37010
UTI,2850,11140,1080,41020,60860,40080,900,10480,17980,70680,25950,55560


In [9]:
top_procedimentos = df_unificado.groupby('descricao_procedimento')['custo_total'] \
    .sum() \
    .sort_values(ascending=False) \
    .reset_index()

top_procedimentos.head(10)

,descricao_procedimento,custo_total
0,Cirurgia,1325000
1,Diária hospitalar,63600
2,Raio-X,43200
3,Consulta médica,34950
4,Exame de sangue,20320


In [10]:
tabela_resumo.style.format({
    'Custo Total': 'R$ {:,.2f}',
    'Tempo Médio Internação': '{:.1f}'
})

,Setor,Custo Total,Qtd Procedimentos,Tempo Médio Internação,Qtd Pacientes
0,Centro Cirúrgico,"R$ 332,360.00",211,8.0,35
1,Emergência,"R$ 507,690.00",382,8.5,46
2,Enfermaria,"R$ 308,440.00",281,7.2,32
3,UTI,"R$ 338,580.00",306,7.0,39


In [11]:
tabela_resumo.to_csv('../data/processed/tabela_resumo.csv', index=False)
tabela_pivot.to_csv('../data/processed/tabela_pivot.csv')